# Multimodal Data Processing with Pixeltable and Backblaze B2

## Extract video frames and store in Backblaze B2

Learn how to process video files with **[Pixeltable](https://www.pixeltable.com/)** and store the results in **[Backblaze B2](https://www.backblaze.com/cloud-storage)**  cloud storage.

**What you'll build:**
- Set up [Pixeltable](https://www.pixeltable.com/) with [Backblaze B2](https://www.backblaze.com/cloud-storage)  integration
- Create a video table and load video files
- Extract frames from video at specific intervals
- Convert frames to grayscale
- Store grayscale frames in Backblaze B2 with automatic URL generation
- (Bonus) Use AI to edit frames with [Reve](https://reve.com/) and store edited images to B2

## About Pixeltable

**[Pixeltable](https://www.pixeltable.com/)** is an open-source AI data infrastructure that provides:
- **Computed Columns:** Automatically process data through AI models and transformations ([docs](https://docs.pixeltable.com/overview/pixeltable))
- **Multimodal Support:** Native handling of images, video, audio, and documents
- **Persistent Storage:** Everything is stored in a database that survives restarts
- **Declarative Storage:** Simply specify where to store data—Pixeltable handles uploads and URL generation

Learn more in the [Pixeltable documentation](https://docs.pixeltable.com/overview/pixeltable).

## About Backblaze B2

**[Backblaze B2](https://www.backblaze.com/cloud-storage)**  is S3-compatible cloud storage that's cost-effective and simple. In this notebook, we use it to store processed outputs like extracted frames and transformed images. Pixeltable automatically detects B2 endpoints when you use `https://s3.{region}.backblazeb2.com/` URLs, making it seamless to integrate B2 into your data pipelines.

Learn more in the **[Backblaze B2 Cloud Storage API Documentation](https://www.backblaze.com/apidocs)**.

**Key benefits of using B2 with Pixeltable:**
- **S3-compatible API** - Works seamlessly with Pixeltable's storage system
- **Cost-effective** - Competitive pricing for cloud storage
- **Simple setup** - Just provide your B2 credentials and Pixeltable handles the rest
- **Automatic URL generation** - Pixeltable generates servable URLs for all stored files

**Prerequisites:** Backblaze B2 account (free tier available), Python 3.10+

## Setup

### Dependencies

This notebook uses `uv` for dependency management. Set up your environment using one of the methods below.

**Option 1: Using uv (Recommended)**

If you have `uv` installed, run this in your terminal before starting Jupyter:

```bash
uv sync --locked
uv run python -m ipykernel install --user --name backblaze --display-name backblaze
uv run jupyter notebook
```

**Option 2: From inside the notebook**

Run the dependency cell below before the B2 setup cell. It installs from `requirements.lock`, which is exported from the checked-in `uv.lock`.


In [ ]:
# Install required packages if this kernel was not started from uv sync --locked.
# This uses the checked-in lock export generated from uv.lock.
%pip install -r requirements.lock


### Set up Backblaze B2

Backblaze B2 is S3-compatible, so the notebook uses the S3-compatible endpoint by default. Configure it with the standard `B2_*` environment variables listed in `.env.example`. Pixeltable automatically detects B2 endpoints when you use `https://s3.{region}.backblazeb2.com/` URLs for destinations.

**Step 1: Get your B2 credentials**
- Go to [Backblaze B2 dashboard](https://secure.backblaze.com/user_signin.htm) -> Account -> Application Keys
- Click "Add a New Application Key"
- Select your bucket with read/write permissions (note the bucket name - you will need it for the destination URLs)
- Copy both values immediately (the applicationKey is only shown once):
  - applicationKeyId
  - applicationKey

**Step 2: Export or enter your B2 settings**. `.env.example` is only a template that lists the variable names; export those variables in your shell or enter them when prompted. Non-interactive runs should export the standard variables before running the notebook. The setup cell validates the endpoint and bucket, then runs a bounded B2 bucket preflight before any frame processing or external AI calls.


In [ ]:
# Configure and preflight Backblaze B2 access
from b2_config import (
    B2ConfigError,
    build_b2_config,
    create_b2_s3_client,
    export_s3_compatible_environment,
    preflight_b2_bucket,
)

print('Enter your Backblaze B2 settings:')
try:
    b2_config = build_b2_config()
    export_s3_compatible_environment(b2_config)
    b2_s3_client = create_b2_s3_client(b2_config)
    print(
        f'Validating B2 access (endpoint: {b2_config.endpoint_url}, '
        f'bucket: {b2_config.bucket_name})'
    )
    preflight_b2_bucket(b2_s3_client, b2_config.bucket_name)
except B2ConfigError as exc:
    raise RuntimeError(f'Backblaze B2 setup failed: {exc}') from exc

B2_REGION = b2_config.region
B2_BUCKET_NAME = b2_config.bucket_name
B2_PUBLIC_URL_BASE = b2_config.public_url_base

print(
    f'Backblaze B2 configured (endpoint: {b2_config.endpoint_url}, '
    f'bucket: {B2_BUCKET_NAME})'
)


### Import libraries

In [ ]:
import pixeltable as pxt
from getpass import getpass
from datetime import datetime
import os

## Create video table

There are two ways to get started with the video table:

**Option A: Replicate from [Pixeltable Cloud](https://www.pixeltable.com/) (Recommended)**

Replicate a pre-configured table from [Pixeltable Cloud](https://www.pixeltable.com/) that already contains the video file. This is the fastest way to get started.

**Option B: Create a new table locally**

Create a new table locally and insert the video file yourself. Use this option if you want to start from scratch or use your own video file.

Choose one of the options below to proceed. Both options are ways to create a Pixeltable named `octo_vid` which we'll use as for local storage and orchestration of transformations.

### Option A: Replicate from [Pixeltable Cloud](https://www.pixeltable.com/)

Replicate the pre-configured table from [Pixeltable Cloud](https://www.pixeltable.com/). This table already contains the video file, so you can skip directly to frame extraction.

- **View the table:** [octo_vid](https://www.pixeltable.com/t/pixeltable:partners/b2/octo_vid)
- **Learn more about replicating:** [Data Sharing Documentation](https://docs.pixeltable.com/notebooks/feature-guides/data-sharing#working-with-replicas)

In [ ]:
# Replicate this table to your local environment
octo_replica = pxt.replicate(
    remote_uri='pxt://pixeltable:partners/b2/octo_vid',
    local_path='local_octo_vid'  # Your local table name
)

Next, you'll create a fresh new table from the replica. Replicas are read-only, so this allows you have a writeable copy with values only.

In [ ]:
# Create a Pixeltable table with the replica as source
octo_vid = pxt.create_table(
    'octo_vid',
    source=octo_replica
)

If you used Option A, skip to the "From video to frames" section below. Otherwise, continue with Option B.


### Option B: Create a new table locally

Create a new table locally and insert your own video file.

In [ ]:
# Create a Pixeltable table with a video column
octo_vid = pxt.create_table(
    'octo_vid',
    {'video': pxt.Video},
    if_exists='replace'
)

Insert octopus.mp4 as a video into our table.


In [ ]:
octo_vid.insert([{'video': 'sources/octopus.mp4'}])

## From video to frames

Now we can see the video, ready for processing:


In [ ]:
octo_vid.collect()

We will extract frames from the video, sampling one frame approximately every 15 seconds.

In [ ]:
from pixeltable.iterators import FrameIterator

# Extract frames approximately every 15 seconds
octo_frames_v = pxt.create_view(
    'octopus_teacher_frames',
    octo_vid,
    iterator=FrameIterator.create(
        video=octo_vid.video,
        fps=0.0667  # Extract 1 frame every 15 seconds (1 / 15)
    ),
    if_exists='replace'
)

Iterators in Pixeltable are table-generating functions - we can see the new table we have created in this view. Remember, we started with a single video in a single row. The iterator shredded the video into frames using our `fps` parameter to specify frames per second. In this view, each row is one of those frames. There is also an implicit join here with the base table so you always keep your context with you. This does not mean that the source video is copied multiple times. Instead, Pixeltable references the same media file across rows.

In [ ]:
octo_frames_v.head()

### Convert frames to grayscale and store in Backblaze B2

Convert each frame to grayscale using Pixeltable's built-in `.convert()` method and store the results in Backblaze B2. This demonstrates how Pixeltable makes image transformations as simple as adding computed columns.

Use the `https://` URL format so Pixeltable automatically detects the B2 endpoint. The destination uses `B2_PUBLIC_URL_BASE` from setup. The pattern is:
```
{B2_PUBLIC_URL_BASE}/your-path/
```

**This is the power of declarative infrastructure:** Instead of writing upload code, managing file paths, and handling errors, you just specify where data should go. Pixeltable orchestrates everything.


In [ ]:
import sys, boto3
print(sys.executable)          # should be .../.venv/bin/python
print("boto3:", boto3.__version__)


In [ ]:
# Convert frames to grayscale and store in B2 - Pixeltable handles upload, versioning, and URL generation
octo_frames_v.add_computed_column(
    frame_bw=octo_frames_v.frame.convert('L'),
    destination=f"{B2_PUBLIC_URL_BASE}/output/frames/"
)

### View the grayscale frames

Let's see the grayscale frames we just created and stored in B2. The `head(5)` method (a shortcut for `limit(5).collect()`) shows the first 5 rows of our view, displaying the grayscale frames from the video.

In [ ]:
octo_frames_v.head(5)

### Query frames with servable URLs

Query the frames to display them along with their servable file URLs from Backblaze B2.


In [ ]:
# Query grayscale frames with their B2 URLs
octo_frames_v.select(
    octo_frames_v.pos,
    octo_frames_v.frame_bw,
    octo_frames_v.frame_bw.fileurl
).collect()

## Bonus: Working with Reve in Pixeltable

Pixeltable's Reve integration lets you call Reve's `create`, `edit`, and `remix` endpoints directly from tables so you can iterate on visuals without leaving your data workflows. We'll use edit to take each frame and edit it with the same prompt. In Pixeltable, you can create unique prompts per row as well.

### Documentation

- [Pixeltable Reve Functions](https://docs.pixeltable.com/sdk/latest/reve#module-pixeltable-functions-reve)
- [Reve API Reference](https://api.reve.com/console/docs)

### Prerequisites

- A Reve account with an API key ([https://app.reve.com/](https://app.reve.com/) → Settings → API Keys)

### Important Notes

- Reve usage incurs costs according to your plan—keep an eye on credits.
- Images you send to Reve leave your environment; avoid uploading sensitive or private data.

### Set up Reve API key

In [ ]:
import os
import getpass

reve_api_key_name = 'REVE' + '_API_KEY'
if reve_api_key_name not in os.environ:
    os.environ[reve_api_key_name] = getpass.getpass('Reve API Key: ')


In [ ]:
from pixeltable.functions import reve

### Edit frames with Reve

Use Reve's `edit` function to transform the grayscale frames into vibrant underwater scenes. This demonstrates how you can integrate AI image editing directly into your Pixeltable workflows, using B2 storage as your generated media destination.

In [ ]:
octo_frames_v.add_computed_column(
    frame_reve=reve.edit(
        octo_frames_v.frame_bw,
        (
            'Convert every image into a majestic underwater scene with vibrant colors, '
            'rich textures, and if present, focus on the octopus. If there is no octopus, '
            'the image should be misty, wistful, and focus on a sense of reflection. '
            'All images should still look realistic photographs, not imaginary or overly magical.'
        )
    ),
    if_exists='replace',
    destination=f"{B2_PUBLIC_URL_BASE}/output/reve/"
)

### View the Reve-edited frames

Compare the original grayscale frames with the Reve-edited versions.

In [ ]:
octo_frames_v.select(octo_frames_v.frame_bw, octo_frames_v.frame_reve).head(5)

### Query Reve-edited frames with servable URLs

View the Reve-edited frames along with their servable file URLs from Backblaze B2.


In [ ]:
# Query Reve-edited frames with their B2 URLs
octo_frames_v.select(
    octo_frames_v.pos,
    octo_frames_v.frame_reve,
    octo_frames_v.frame_reve.fileurl
).collect()